In [1]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name of step function
str_name = 'step-genxii-pd-pre-boto3'

### Write ```definition.json```

In [3]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "MakeDFs",
  "States": {
    "MakeDFs": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "job-genxii-pd-make-dfs",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-pd-make-dfs-1",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-pd-make-dfs-1:6"
      },
      "Next": "GetListCols"
    },
    "GetListCols": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "job-genxii-pd-get-list-cols",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-pd-listcols-1:6",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-pd-listcols-1"
      },
      "Next": "EDA"
    },
    "EDA": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "job-genxii-pd-eda",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-pd-eda-1:6",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-pd-eda-1"
      },
      "End": true
    }
  }
}

Writing definition.json


### Make string definition

In [4]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

### Create state machine

In [5]:
cls_client_sfn = boto3.client('stepfunctions')

In [6]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [7]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'genxii-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxii-payload-parsing',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-pd-lambda-boto3',
 'step-genxii-ad-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-feat-select-boto3',
 'step-genxii-ad-model-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3',
 'step-genxii-ad-pre-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-pre-boto

In [8]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine step-genxii-pd-pre-boto3 exists, it will be deleted
Deleting arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-pd-pre-boto3

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Tue, 23 Apr 2024 16:03:13 GMT',
                                      'x-amzn-requestid': '28b67306-b60a-4990-9e43-52e8bc66ef18'},
                      'HTTPStatusCode': 200,
                      'RequestId': '28b67306-b60a-4990-9e43-52e8bc66ef18',
                      'RetryAttempts': 0}}


In [9]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '129',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Tue, 23 Apr 2024 16:04:25 GMT',
                                      'x-amzn-requestid': '78c1136d-6ed6-442c-b7b2-d7867fd51f2b'},
                      'HTTPStatusCode': 200,
                      'RequestId': '78c1136d-6ed6-442c-b7b2-d7867fd51f2b',
                      'RetryAttempts': 4},
 'creationDate': datetime.datetime(2024, 4, 23, 16, 4, 25, 922000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-pd-pre-boto3'}


### Describe state machine

In [10]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-pd-pre-boto3
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1699',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Tue, 23 Apr 2024 16:04:25 GMT',
                                      'x-amzn-requestid': '85f2b04c-0c8e-4ae2-9f29-981191057702'},
                      'HTTPStatusCode': 200,
                      'RequestId': '85f2b04c-0c8e-4ae2-9f29-981191057702',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 4, 23, 16, 4, 25, 922000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"MakeDFs", "States": {"MakeDFs": {"Type": "Task", "Resource": '
               '"arn:aws:states:::batch:submitJob.sync", "Parameters": '
               '{"JobName": "job-

### Execute step function workflow

In [11]:
# # start execution
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )

### Clean-up

In [12]:
os.remove('./definition.json')